# Steganography GAN — train on a cloud GPU (Colab / similar)

**What this does:** installs a CUDA build of PyTorch, fetches this project, runs  
`scripts/train_production_gan_gpu.py` (the **app-compatible** trainer — not the old `colab_train_gpu.py` at repo root).

1. In Colab: **Runtime → Change runtime type → GPU** (T4 is fine; A100 is faster if available).
2. If your project is only on your laptop: use **File → Upload** to your Drive (see next cells) or push to GitHub and set `REPO_URL` below.
3. Checkpoints are saved to `models/checkpoints/*/best_model.pth` under the repo. Copy with Google Drive (optional cell) or **download** from the Files pane.
4. If you ever **re-pip** a different `torch` in the same session, **do not** `importlib.reload` it — use **Runtime → Restart session** once, then continue (mixing two torch builds in one process breaks the native `TORCH_LIBRARY` registry).
5. If you hit **`TORCH_LIBRARY` / `triton` errors** after a bad re-install: **Runtime → Restart session** (wipes a broken in-memory torch), then re-run the updated cell 1.
6. If **CUDA is `False`:** this cell will `pip install` a CUDA wheel, then **exit**; **Restart** and re-run; on a **GPU** runtime, preinstalled `torch` usually has CUDA and no extra `pip` is needed.

In [ ]:
# 1) Deps: keep Colab’s preinstalled torch if it already has CUDA (do NOT pip another wheel + reload — that crashes).
import subprocess, sys, os

def sh(cmd: str) -> int:
    print("$", cmd)
    return subprocess.call(cmd, shell=True)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Colab 2024+ often ships `torch` + CUDA already. Do not pip a second wheel in the *same* session and reload (native registry error).
if not torch.cuda.is_available():
    sh("pip install -q 'torch' 'torchvision' 'torchaudio' --index-url https://download.pytorch.org/whl/cu124")
    print("\n==========")
    print("Required: **Runtime → Restart session**, then run this cell again.")
    print("(A fresh process loads the new torch once — do not `reload` it.)")
    print("==========\n")
    import sys as _sys
    _sys.exit(0)  # stop; continue after restart
else:
    print("Using existing PyTorch+CUDA. Skipping torch re-install.\n")

sh("pip install -q cryptography pillow numpy scipy")
print("final — cuda available:", torch.cuda.is_available(), "| torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Get the project: choose ONE of (A) git clone, (B) Google Drive, (C) Colab file upload to /content

import os, subprocess, zipfile, shutil, pathlib
WORKDIR = "/content/stego_project"  # change if you use another path

# (A) Public git repo (replace with your URL). Leave empty to skip.
# The URL must be a Python *string* — use "double quotes" around the whole URL.
REPO_URL = "https://github.com/namanp09/steganography-project.git"  # or "" to clone via zip/Drive instead

if REPO_URL.strip() and not os.path.isdir(pathlib.Path(WORKDIR) / ".git"):
    p = pathlib.Path(WORKDIR).parent
    p.mkdir(parents=True, exist_ok=True)
    if os.path.isdir(WORKDIR):
        shutil.rmtree(WORKDIR)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, WORKDIR])
    print("Cloned to:", WORKDIR)
elif os.path.isdir(pathlib.Path(WORKDIR) / ".git") or os.path.isfile(pathlib.Path(WORKDIR) / "scripts" / "train_production_gan_gpu.py"):
    print("Using existing project at:", WORKDIR)
else:
    # (B) After uploading a zip to Drive, set ZIP_PATH, or (C) upload a zip in Colab file browser to /content and set name
    DRIVE_ZIP = ""  # e.g. "/content/drive/MyDrive/steganography-project.zip" after mounting Drive (next cell)
    UPLOADED = "/content/steganography-project.zip"  # or upload the repo zip to /content and keep this name
    for candidate in (DRIVE_ZIP, UPLOADED):
        if candidate and os.path.isfile(candidate):
            extract_to = str(pathlib.Path(WORKDIR).parent)
            with zipfile.ZipFile(candidate, "r") as z:
                z.extractall(extract_to)
            # if zip contains one top folder, set WORKDIR to that folder; adjust as needed
            kids = [os.path.join(extract_to, x) for x in os.listdir(extract_to) if not x.startswith(".")]
            dirs = [k for k in kids if os.path.isdir(k) and (pathlib.Path(k) / "scripts" / "train_production_gan_gpu.py").is_file()]
            if len(dirs) == 1:
                WORKDIR = dirs[0]
            print("Extracted; set WORKDIR =", WORKDIR)
            break
    else:
        print("Set REPO_URL to a git URL, or upload a zip to /content/steganography-project.zip and re-run, or mount Drive and set DRIVE_ZIP.")

In [ ]:
# 2b) **If cell 4 says "Old script (no --log-file)"** — the notebook is new but Colab’s *Python files* are old.\n
#    Updating the .ipynb in Colab does NOT update `scripts/train_production_gan_gpu.py` on disk.\n
import os, subprocess, sys, urllib.request
W = "/content/stego_project"  # same as WORKDIR in cell 2
if os.path.isdir(W + "/.git"):
    subprocess.call(["git", "-C", W, "pull"], env={**os.environ})
else:
    print("No git repo; downloading single file from GitHub (main branch)…")
    u = "https://raw.githubusercontent.com/namanp09/steganography-project/main/scripts/train_production_gan_gpu.py"
    out = W + "/scripts/train_production_gan_gpu.py"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    urllib.request.urlretrieve(u, out)
    print("Saved", out, "→ re-run cell 4.")
h = subprocess.run([sys.executable, W + "/scripts/train_production_gan_gpu.py", "--help"], capture_output=True, text=True, cwd=W)
ok = h.returncode == 0 and "--log-file" in (h.stdout or "")
print("train_production_gan_gpu.py has --log-file:", ok, "(if False, push latest from your PC to GitHub, then re-run 2b)")

In [ ]:
# 3) Optional: mount Google Drive to save checkpoints outside Colab (persist after session ends)
USE_DRIVE = False  # set True to mount and copy final checkpoint in last cell

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
else:
    print("Skipping Drive. Enable USE_DRIVE to persist outputs on Drive.")

In [ ]:
# 4) Run training: checks if script has --log-file (old clones skip it; run: !cd $WORKDIR && git pull). Uses python -u + unbuffered env for streaming.
import os, sys, pathlib

if "WORKDIR" not in dir() or not os.path.isfile(pathlib.Path(WORKDIR) / "scripts" / "train_production_gan_gpu.py"):
    WORKDIR = "/content/stego_project"  # adjust to your path
if not os.path.isfile(pathlib.Path(WORKDIR) / "scripts" / "train_production_gan_gpu.py"):
    raise SystemExit(f"Not found: {WORKDIR}/scripts/train_production_gan_gpu.py — fix cell 2.")

EPOCHS = 200
SAMPLES = 8000  # increase if you have time (better generalization; longer runs)
BATCH = 32      # lower to 16 if CUDA OOM (e.g. T4 16GB should be ok for image at default sizes)
MODALITY = "image"  # or "video" or "audio"

os.chdir(WORKDIR)
import subprocess
print("cwd =", os.getcwd(), flush=True)
LOG = os.path.join(WORKDIR, "train.log")
h = subprocess.run([sys.executable, "scripts/train_production_gan_gpu.py", "--help"], capture_output=True, text=True, cwd=WORKDIR)
HAS_LOG = h.returncode == 0 and "--log-file" in (h.stdout or "")
if HAS_LOG:
    print("--log-file supported → mirroring to:", LOG)
else:
    print('Old script (no --log-file). In a new cell:  !cd ' + WORKDIR + ' && git pull   then re-run 4.')
print("Tip: !tail -n 25" + (" " + LOG if HAS_LOG else "  (after git pull)") + "   or  !nvidia-smi")
print("tqdm: (1) all epochs, (2) batches this epoch.\n")
cmd = [
    sys.executable, "-u", "scripts/train_production_gan_gpu.py",
    "--modality", MODALITY, "--device", "cuda",
    "--epochs", str(EPOCHS), "--train-samples", str(SAMPLES), "--batch", str(BATCH),
]
if HAS_LOG:
    cmd += ["--log-file", LOG]
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
r = subprocess.call(cmd, cwd=WORKDIR, env=env)
if r != 0:
    raise SystemExit(f"Training exited with {r}")

In [ ]:
# 5) Optional: copy best checkpoint to Google Drive (enable USE_DRIVE in cell 3 first)
if "USE_DRIVE" not in dir():
    USE_DRIVE = False
if "MODALITY" not in dir():
    MODALITY = "image"
if "WORKDIR" not in dir():
    WORKDIR = "/content/stego_project"
if USE_DRIVE:
    import shutil, pathlib, glob
    from pathlib import Path
    ck = Path(WORKDIR) / "models" / "checkpoints" / ("image_gan_improved" if MODALITY=="image" else ("video_gan_improved" if MODALITY=="video" else "audio_gan_improved")) / "best_model.pth"
    if ck.is_file():
        dest = Path("/content/drive/MyDrive/stego_checkpoints")
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ck, dest / ck.name)
        print("Saved to", dest / ck.name)
    else:
        print("Checkpoint not found at", ck)
else:
    print("Download models/checkpoints/ from Colab’s Files pane, or enable USE_DRIVE in cell 3 and re-run this cell.")